# 🚀 Ultimate 3D Multi-Modal Tracklet Pipeline (v11)
This is the complete, end-to-end solution for object classification using the KITTI data format. It fuses **3D LiDAR tracklet geometry** with **2D Camera visual features** and includes a comprehensive visualization suite.

### Pipeline Overview:
1. **Preprocessing**: Converts raw XML tracklets into a standardized format.
2. **Calibration Engine**: Calculates the Projection Matrix to bridge 3D and 2D spaces.
3. **Vision Branch**: Uses a pretrained Faster R-CNN ResNet-50 FPN to extract visual context.
4. **Fusion**: Concatenates spatial and visual features into temporal sequences.
5. **Temporal Model**: A Bidirectional LSTM (Bi-LSTM) classifies the 32-frame sequence.
6. **Evaluation & Visualization**: Metrics (mAP), Confusion Matrix, Performance Curves, and Inference Overlays.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# --- GLOBAL CONFIGURATION ---
TARGET_FRAMES = 32
COORD_FEATURES = 9
VISUAL_FEATURES = 1024
NUM_CLASSES = 5
CLASS_MAP = {'Car': 0, 'Van': 1, 'Truck': 2, 'Pedestrian': 3, 'Cyclist': 4}
INV_CLASS_MAP = {v: k for k, v in CLASS_MAP.items()}

## 1. Calibration Engine
Projects 3D LiDAR space ($tx, ty, tz$) into 2D pixel space ($u, v$).

In [ ]:
def get_projection_matrix(cam_to_cam_path, velo_to_cam_path):
    with open(velo_to_cam_path, 'r') as f:
        data = f.readlines()
        R = np.array([float(x) for x in data[1].split()[1:]]).reshape(3, 3)
        T = np.array([float(x) for x in data[2].split()[1:]]).reshape(3, 1)
        tr_velo_to_cam = np.vstack([np.hstack([R, T]), [0, 0, 0, 1]])

    with open(cam_to_cam_path, 'r') as f:
        data = f.readlines()
        r_rect = np.eye(4)
        r_rect[:3, :3] = np.array([float(x) for x in data[8].split()[1:]]).reshape(3, 3)
        p_rect_02 = np.array([float(x) for x in data[25].split()[1:]]).reshape(3, 4)

    return p_rect_02 @ r_rect @ tr_velo_to_cam

def project_3d_to_2d(p_matrix, x, y, z):
    pt_3d = np.array([x, y, z, 1.0])
    pt_2d = p_matrix @ pt_3d
    if pt_2d[2] == 0: return 0, 0
    return int(pt_2d[0] / pt_2d[2]), int(pt_2d[1] / pt_2d[2])

## 2. Vision Feature Extractor
Extracts visual descriptors using Faster R-CNN FPN backbone.

In [ ]:
class VisionExtractor:
    def __init__(self):
        full_model = fasterrcnn_resnet50_fpn(pretrained=True)
        self.backbone = full_model.backbone
        self.backbone.eval()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.backbone.to(self.device)
        self.transform = T.Compose([
            T.Resize((224, 224)), T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def extract(self, pil_img):
        img_t = self.transform(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            feats = self.backbone(img_t)
            return torch.mean(feats['0'], dim=[2, 3]).cpu().numpy().flatten()

## 3. Temporal Model Architecture

In [ ]:
def build_model():
    model = models.Sequential([
        layers.Input(shape=(TARGET_FRAMES, COORD_FEATURES + VISUAL_FEATURES)),
        layers.Conv1D(128, kernel_size=3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

## 4. Performance Visualizations
Includes Loss/Accuracy curves and the Confusion Matrix.

In [ ]:
def plot_training_metrics(history):
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    # Accuracy
    ax[0].plot(history.history['accuracy'], label='Train')
    ax[0].plot(history.history['val_accuracy'], label='Val')
    ax[0].set_title('Model Accuracy'); ax[0].legend()
    # Loss
    ax[1].plot(history.history['loss'], label='Train')
    ax[1].plot(history.history['val_loss'], label='Val')
    ax[1].set_title('Model Loss'); ax[1].legend()
    plt.show()

def plot_cm(y_true, y_pred):
    y_true_labels = np.argmax(y_true, axis=1)
    y_pred_labels = np.argmax(y_pred, axis=1)
    cm = confusion_matrix(y_true_labels, y_pred_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(CLASS_MAP.keys()))
    disp.plot(cmap='Blues'); plt.title("Confusion Matrix"); plt.show()

## 5. Visual Inference Overlays
Project predictions directly onto images for verification.

In [ ]:
def visualize_inference(img_path, p_matrix, tx, ty, tz, pred_idx, true_idx):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    u, v = project_3d_to_2d(p_matrix, tx, ty, tz)
    
    # Draw Projection Point
    cv2.circle(img, (u, v), 8, (255, 0, 0), -1)
    
    # Label Display
    color = (0, 255, 0) if pred_idx == true_idx else (255, 0, 0)
    label_text = f"Pred: {INV_CLASS_MAP[pred_idx]} | True: {INV_CLASS_MAP[true_idx]}"
    cv2.putText(img, label_text, (u - 60, v - 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    plt.figure(figsize=(10, 6)); plt.imshow(img); plt.axis('off'); plt.show()